# Single-Phase Shock Tube — SCMM (Gessner 2010) vs Godunov Finite Volume

Phase-1/Phase-2 benchmark notebook. Reproduces the single-phase shock-tube
problem of Gessner (2010), Sec. 3.4 (air, 500 kPa/400 K vs 100 kPa/300 K,
4 m tube), with:

* the reconstructed **Split Coefficient Matrix Method** (conservative and
  non-conservative formulations, 1st/2nd order), and
* **Godunov-type finite-volume** solvers (exact/HLL/HLLC fluxes, first-order
  and MUSCL-Hancock),

validated against the **exact Riemann solution** and the thesis error table.

> Runtime note: the full thesis resolution (4001 nodes, Δt = 1 µs) takes
> several minutes per SCMM run. Set `FULL_RESOLUTION = False` for a fast
> (~10 s) coarse-grid pass with the same physics.


In [ ]:
# Install the package from GitHub (Colab) -- or use a local clone
try:
    import hyperbolic1d
except ImportError:
    %pip install -q "hyperbolic1d[plot] @ git+https://github.com/USER/REPO"
    import hyperbolic1d
print(hyperbolic1d.__version__)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from hyperbolic1d import solve_scmm_shock_tube, solve_godunov_shock_tube
from hyperbolic1d.benchmarks import (GESSNER_SINGLE_PHASE, THESIS_TABLE_3_3,
                                     exact_reference, mean_abs_errors)

FULL_RESOLUTION = False   # True = thesis grid (4001 nodes, dt = 1 us)

if FULL_RESOLUTION:
    scmm_kw = dict(n_nodes=4001, dt=1e-6, t_end=3e-3)
    n_cells = 4000
else:
    scmm_kw = dict(n_nodes=801, dt=5e-6, t_end=3e-3)
    n_cells = 800

## 1. Exact solution (validation reference)

In [ ]:
x = np.linspace(0, 4, 2001)
ref = exact_reference(x, 3e-3)
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (name, label) in zip(axes.flat, [("p","Pressure [Pa]"),("u","Velocity [m/s]"),
                                         ("rho","Density [kg/m$^3$]"),("T","Temperature [K]")]):
    ax.plot(x, ref[name], 'k-', lw=1.5)
    ax.set_xlabel("x [m]"); ax.set_ylabel(label); ax.grid(alpha=0.3)
fig.suptitle("Exact Riemann solution at t = 3 ms"); fig.tight_layout()

## 2. SCMM runs (Phase 1 reconstruction)

In [ ]:
scmm = {}
for form in ("nonconservative", "conservative"):
    for order in (1, 2):
        key = f"{form[:7]}-o{order}"
        scmm[key] = solve_scmm_shock_tube(formulation=form, order=order, **scmm_kw)
        print(key, mean_abs_errors(scmm[key]), f"{scmm[key].wall_time:.1f}s")

Compare with thesis Table 3.3 (full resolution only):

| Formulation | P [Pa] | U [m/s] | ρ [kg/m³] | T [K] |
|---|---|---|---|---|
| Non-conservative, 1st | 3339 | 4.65 | 0.0253 | 2.14 |
| Non-conservative, 2nd | 2713 | 3.87 | 0.0201 | 1.67 |
| Conservative, 1st | 1555 | 2.02 | 0.0161 | 1.74 |
| Conservative, 2nd | 311.6 | 0.331 | 0.00354 | 0.310 |

## 3. Godunov finite-volume runs (Phase 2)

In [ ]:
fv = {}
for key, kw in {
    "FV-o1-HLLC":  dict(flux="hllc", reconstruction="constant"),
    "FV-MUSCL-HLLC": dict(flux="hllc", reconstruction="muscl", limiter="vanleer"),
}.items():
    fv[key] = solve_godunov_shock_tube(n_cells=n_cells, **kw)
    e = mean_abs_errors(fv[key])
    print(key, e, f"{fv[key].wall_time:.2f}s",
          "mass defect: %.1e" % ((fv[key].mass_final-fv[key].mass_initial)/fv[key].mass_initial))

## 4. Field comparison

In [ ]:
sel = {"SCMM cons. 2nd order": scmm["conserv-o2"],
       "SCMM non-cons. 2nd order": scmm["noncons-o2"],
       "FV 1st order HLLC": fv["FV-o1-HLLC"],
       "FV MUSCL-Hancock HLLC": fv["FV-MUSCL-HLLC"]}
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (name, label) in zip(axes.flat, [("p","Pressure [Pa]"),("u","Velocity [m/s]"),
                                         ("rho","Density [kg/m$^3$]"),("T","Temperature [K]")]):
    ax.plot(x, ref[name], 'k-', lw=2, label="exact")
    for lab, r in sel.items():
        ax.plot(r.x, getattr(r, name if name!="p" else "p"), lw=1, label=lab)
    ax.set_xlabel("x [m]"); ax.set_ylabel(label); ax.grid(alpha=0.3)
axes[0,0].legend(fontsize=8)
fig.suptitle(f"Shock tube at t = 3 ms"); fig.tight_layout()

In [ ]:
# zoom on the shock front
i0 = np.argmax(ref["u"] > 1)   # approximate wave region
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(x, ref["rho"], 'k-', lw=2, label="exact")
for lab, r in sel.items():
    ax.plot(r.x, r.rho, '.-', ms=2, lw=0.7, label=lab)
ax.set_xlim(3.0, 3.8); ax.set_xlabel("x [m]"); ax.set_ylabel("Density [kg/m$^3$]")
ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_title("Contact + shock region (zoom)")

## 5. Error and cost summary

Mean absolute errors against the exact solution, wall time, and (for FV)
discrete conservation defects. Use this table — not prior expectations — to
judge which architecture carries forward to the wellbore model.

In [ ]:
import pandas as pd
rows = []
for lab, r in {**{k:v for k,v in scmm.items()}, **fv}.items():
    e = mean_abs_errors(r)
    row = dict(scheme=lab, **{k: e[k] for k in ("P","U","rho","T")},
               wall_s=round(r.wall_time,2))
    if hasattr(r, "mass_final"):
        row["mass_defect"] = (r.mass_final-r.mass_initial)/r.mass_initial
    rows.append(row)
pd.DataFrame(rows).set_index("scheme")